In [4]:
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [5]:
!pip install kaggle -q

import os
os.makedirs('/content/cassava', exist_ok=True)

!kaggle competitions download -c cassava-leaf-disease-classification -p /content/cassava

100% 5.76G/5.76G [00:57<00:00, 107MB/s]



In [6]:
import zipfile

with zipfile.ZipFile('/content/cassava/cassava-leaf-disease-classification.zip', 'r') as z:
    z.extractall('/content/cassava')

print("Done extracting")
!ls /content/cassava

Done extracting
cassava-leaf-disease-classification.zip  test_images	 train_images
label_num_to_disease_map.json		 test_tfrecords  train_tfrecords
sample_submission.csv			 train.csv


In [7]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
Device: Tesla T4


In [8]:
BASE_DIR = '/content/cassava'
IMG_DIR = os.path.join(BASE_DIR, 'train_images')

df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))

with open(os.path.join(BASE_DIR, 'label_num_to_disease_map.json')) as f:
    label_map = json.load(f)

print(f"Total samples: {len(df)}")
print(f"Class distribution:\n{df['label'].value_counts().sort_index()}")
print(f"\nLabel map: {label_map}")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\nTrain size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"\nTrain class distribution:\n{train_df['label'].value_counts().sort_index()}")
print(f"\nVal class distribution:\n{val_df['label'].value_counts().sort_index()}")

Total samples: 21397
Class distribution:
label
0     1087
1     2189
2     2386
3    13158
4     2577
Name: count, dtype: int64

Label map: {'0': 'Cassava Bacterial Blight (CBB)', '1': 'Cassava Brown Streak Disease (CBSD)', '2': 'Cassava Green Mottle (CGM)', '3': 'Cassava Mosaic Disease (CMD)', '4': 'Healthy'}

Train size: 17117
Val size: 4280

Train class distribution:
label
0      870
1     1751
2     1909
3    10526
4     2061
Name: count, dtype: int64

Val class distribution:
label
0     217
1     438
2     477
3    2632
4     516
Name: count, dtype: int64


In [9]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df['image_id'][idx])
        image = Image.open(img_path).convert('RGB')
        label = self.df['label'][idx]
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = CassavaDataset(train_df, IMG_DIR, transform=train_transforms)
val_dataset = CassavaDataset(val_df, IMG_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 535
Val batches: 134


In [10]:
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")
print(f"Label values in batch: {labels.unique()}")
print(f"Image dtype: {images.dtype}")
print(f"Min pixel val: {images.min():.3f}, Max pixel val: {images.max():.3f}")

Image batch shape: torch.Size([32, 3, 224, 224])
Labels batch shape: torch.Size([32])
Label values in batch: tensor([0, 1, 2, 3, 4])
Image dtype: torch.float32
Min pixel val: -2.118, Max pixel val: 2.640
